In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

In [3]:
from pathlib import Path
import json
import stat
import platform

from tqdm import tqdm
from rich.pretty import pprint
import paramiko

from enderleaf.const import (
    PT_ADDRESSES,
    PT_OUTPUT,
    PT_IMAGES_RECEIVED,
    PT_IMAGES_VALIDATED,
)
from enderleaf.tools import ensure_folder, list_remote_files, list_remote_folders, read_dataframe
from enderleaf.image import load_image

In [4]:
with open(str(PT_ADDRESSES), "r") as file:
    addreesses = json.load(file)

addreesses

{'nodes': ['147.100.144.150', '147.100.144.195'], 'main': '147.100.144.30'}

In [6]:
platform.system()

'Darwin'

In [ ]:
ssh = paramiko.SSHClient()
pk_file = Path.home().joinpath(".ssh/id_ed25519")
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(
    addreesses["nodes"][0],
    username="ender",
    pkey=paramiko.Ed25519Key.from_private_key_file(pk_file),
)
ssh.close()

In [ ]:
# ssh = paramiko.SSHClient()
# try:
#     ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
#     pk_file = Path.home().joinpath(".ssh/id_ed25519")
#     ssh.connect(
#         addreesses["nodes"][0],
#         username="ender",
#         pkey=paramiko.Ed25519Key.from_private_key_file(pk_file),
#     )
#     sftp = ssh.open_sftp()
#     try:
#         src_folders = list_remote_folders(sftp, "/home/ender/code/EnderLeaf/output")
#         for src_fld in src_folders:
#             dst_fld = PT_OUTPUT.joinpath(
#                 *src_fld.parts[src_fld.parts.index("output") + 1 :]
#             )
#             ensure_folder(dst_fld)
#         src_files = list_remote_files(sftp, "/home/ender/code/EnderLeaf/output")
#         for src_file in tqdm(src_files):
#             dst_file = PT_OUTPUT.joinpath(
#                 *src_file.parts[src_file.parts.index("output") + 1 :]
#             )
#             if (
#                 dst_file.is_file()
#                 and dst_file.stat().st_size == sftp.stat(str(src_file)).st_size
#             ):
#                 continue
#             sftp.get(str(src_file), str(dst_file))
#     except Exception as e:
#         print(str(e))
#     finally:
#         sftp.close()
# except Exception as e:
#     print(str(e))
# finally:
#     ssh.close()

In [ ]:
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
pk_file = Path.home().joinpath(".ssh/id_ed25519")
ssh.connect(
    addreesses["nodes"][0],
    username="ender",
    pkey=paramiko.Ed25519Key.from_private_key_file(pk_file),
)
sftp = ssh.open_sftp()
src_folders = list_remote_folders(sftp, "/home/ender/code/EnderLeaf/output")
for src_fld in src_folders:
    ensure_folder(
        PT_IMAGES_RECEIVED.joinpath(*src_fld.parts[src_fld.parts.index("output") + 1 :])
    )
    ensure_folder(
        PT_IMAGES_VALIDATED.joinpath(
            *src_fld.parts[src_fld.parts.index("output") + 1 :]
        )
    )
src_files = list_remote_files(sftp, "/home/ender/code/EnderLeaf/output")
to_delete_files = []
progress = tqdm(range(len(src_files)), desc="Toto")
for idx in progress:
    src_file = src_files[idx]
    received_file = PT_IMAGES_RECEIVED.joinpath(
        *src_file.parts[src_file.parts.index("output") + 1 :]
    )
    validated_file = PT_IMAGES_VALIDATED.joinpath(
        *src_file.parts[src_file.parts.index("output") + 1 :]
    )
    progress.set_description(
        f"Recived files: {len(list(PT_IMAGES_RECEIVED.rglob('*.*')))} | Validated files: {len(list(PT_IMAGES_VALIDATED.rglob('*.*')))} | To be deleted {len(to_delete_files)}"
    )
    if not validated_file.is_file() and (
        not received_file.is_file()
        or received_file.stat().st_size != sftp.stat(str(src_file)).st_size
    ):
        sftp.get(str(src_file), str(received_file))
        if received_file.stat().st_size != sftp.stat(str(src_file)).st_size:
            received_file.unlink()
            continue
        try:
            if received_file.suffix == ".csv":
                read_dataframe(received_file)
            elif received_file.suffix == ".png":
                load_image(received_file)
        except Exception as e:
            received_file.unlink()
            continue
        else:
            received_file.rename(validated_file)
    if (
        validated_file.is_file()
        and validated_file.stat().st_size == sftp.stat(str(src_file)).st_size
    ):
        to_delete_files.append(src_files)
# for d_file in to_delete_files:
#     sftp.remove(d_file)
sftp.close()

print(len(to_delete_files))